In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox
import random
import heapq
from collections import deque

# Hằng số cho thuật toán IDS
CUTOFF = "CUTOFF"
FAILURE = "FAILURE"

class VacuumGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("Mô phỏng AI Máy Hút Bụi")
        self.root.geometry("1000x600")

        # Biến trạng thái
        self.m = 4
        self.n = 4
        self.room = []
        self.start_x = 0
        self.start_y = 0
        self.current_x = 0
        self.current_y = 0
        self.is_animating = False
        self.animation_id = None
        self.path = []
        self.step_idx = 0

        self.setup_ui()
        self.generate_room()

    def setup_ui(self):
        # Bố cục chính
        self.root.columnconfigure(1, weight=1)
        self.root.rowconfigure(0, weight=1)

        # === BÊN TRÁI: Bảng điều khiển ===
        left_frame = tk.Frame(self.root, width=200, bg="#f0f0f0", padx=10, pady=10)
        left_frame.grid(row=0, column=0, sticky="ns")
        left_frame.grid_propagate(False)

        tk.Label(left_frame, text="KÍCH THƯỚC SÀN", bg="#f0f0f0", font=("Arial", 10, "bold")).pack(pady=(0, 5))

        row_frame = tk.Frame(left_frame, bg="#f0f0f0")
        row_frame.pack(fill="x", pady=2)
        tk.Label(row_frame, text="Số dòng (m):", bg="#f0f0f0").pack(side="left")
        self.entry_m = tk.Entry(row_frame, width=5)
        self.entry_m.insert(0, "4")
        self.entry_m.pack(side="right")

        col_frame = tk.Frame(left_frame, bg="#f0f0f0")
        col_frame.pack(fill="x", pady=2)
        tk.Label(col_frame, text="Số cột (n):", bg="#f0f0f0").pack(side="left")
        self.entry_n = tk.Entry(col_frame, width=5)
        self.entry_n.insert(0, "4")
        self.entry_n.pack(side="right")

        self.btn_generate = tk.Button(left_frame, text="Tạo sàn mới", command=self.generate_room, bg="#d9edf7")
        self.btn_generate.pack(fill="x", pady=10)

        tk.Label(left_frame, text="THUẬT TOÁN", bg="#f0f0f0", font=("Arial", 10, "bold")).pack(pady=(15, 5))
        self.algo_var = tk.StringVar(value="BFS Loại 1")
        # Đã thêm 3 thuật toán mới vào Combobox
        self.cb_algo = ttk.Combobox(left_frame, textvariable=self.algo_var, state="readonly",
                                    values=["BFS Loại 1", "BFS Loại 2",
                                            "DFS Loại 1", "DFS Loại 2",
                                            "IDS Loại 1", "IDS Loại 2",
                                            "Uniform Cost Search (UCS)"])
        self.cb_algo.pack(fill="x", pady=5)

        self.btn_start = tk.Button(left_frame, text="Bắt đầu", command=self.start_simulation, bg="#dff0d8", font=("Arial", 10, "bold"))
        self.btn_start.pack(fill="x", pady=10)

        self.btn_stop = tk.Button(left_frame, text="Kết thúc", command=self.stop_simulation, bg="#f2dede")
        self.btn_stop.pack(fill="x", pady=5)

        # === Ở GIỮA: Sàn nhà ===
        center_frame = tk.Frame(self.root, bg="white", bd=2, relief="sunken")
        center_frame.grid(row=0, column=1, sticky="nsew", padx=10, pady=10)

        self.canvas = tk.Canvas(center_frame, bg="white")
        self.canvas.pack(fill="both", expand=True)
        # Bind sự kiện resize để vẽ lại grid cho vừa
        self.canvas.bind("<Configure>", lambda e: self.draw_grid() if self.room else None)

        # === BÊN PHẢI: Log hệ thống ===
        right_frame = tk.Frame(self.root, width=250)
        right_frame.grid(row=0, column=2, sticky="ns", padx=10, pady=10)
        right_frame.grid_propagate(False)

        tk.Label(right_frame, text="LOG CÁC BƯỚC CHẠY", font=("Arial", 10, "bold")).pack()
        self.txt_log = tk.Text(right_frame, width=30, state="disabled", font=("Courier", 9))
        scrollbar = tk.Scrollbar(right_frame, command=self.txt_log.yview)
        self.txt_log.config(yscrollcommand=scrollbar.set)
        scrollbar.pack(side="right", fill="y")
        self.txt_log.pack(side="left", fill="both", expand=True)

        # === BÊN DƯỚI: Kết quả đường đi ===
        bottom_frame = tk.Frame(self.root, height=120, bg="#e8e8e8", bd=2, relief="groove")
        bottom_frame.grid(row=1, column=0, columnspan=3, sticky="ew")
        bottom_frame.pack_propagate(False)

        tk.Label(bottom_frame, text="KẾT QUẢ ĐƯỜNG ĐI:", font=("Arial", 10, "bold"), bg="#e8e8e8").pack(anchor="w", padx=10, pady=(5,0))
        self.lbl_result = tk.Label(bottom_frame, text="Chưa có dữ liệu.", font=("Arial", 10), bg="#e8e8e8", fg="blue", wraplength=950, justify="left")
        self.lbl_result.pack(anchor="w", padx=10, fill="x")

    def log(self, message):
        self.txt_log.config(state="normal")
        self.txt_log.insert(tk.END, message + "\n")
        self.txt_log.see(tk.END)
        self.txt_log.config(state="disabled")

    def clear_log(self):
        self.txt_log.config(state="normal")
        self.txt_log.delete(1.0, tk.END)
        self.txt_log.config(state="disabled")

    def generate_room(self):
        self.stop_simulation()
        try:
            self.m = int(self.entry_m.get())
            self.n = int(self.entry_n.get())
            if self.m <= 0 or self.n <= 0: raise ValueError
        except ValueError:
            messagebox.showerror("Lỗi nhập liệu", "Kích thước m, n phải là số nguyên dương.")
            return

        # Khởi tạo phòng toàn rác (1)
        self.room = [[1 for _ in range(self.n)] for _ in range(self.m)]

        # Đặt ngẫu nhiên máy hút bụi
        self.start_x = random.randint(0, self.m - 1)
        self.start_y = random.randint(0, self.n - 1)
        self.room[self.start_x][self.start_y] = 0

        self.current_x, self.current_y = self.start_x, self.start_y

        self.clear_log()
        self.log(f"Đã tạo sàn {self.m}x{self.n}")
        self.log(f"Vị trí bắt đầu: ({self.start_x}, {self.start_y})")
        self.lbl_result.config(text="Sẵn sàng...", fg="black")

        self.draw_grid()

    def draw_grid(self):
        self.canvas.delete("all")
        if not self.room: return

        c_width = self.canvas.winfo_width()
        c_height = self.canvas.winfo_height()
        # Tránh lỗi khi canvas chưa load xong
        if c_width <= 1: c_width = 400
        if c_height <= 1: c_height = 400

        cell_w = c_width / self.n
        cell_h = c_height / self.m

        for i in range(self.m):
            for j in range(self.n):
                x1, y1 = j * cell_w, i * cell_h
                x2, y2 = x1 + cell_w, y1 + cell_h

                # Màu nền: Xám nếu bẩn (1), Trắng nếu sạch (0)
                color = "#d3d3d3" if self.room[i][j] == 1 else "#ffffff"
                self.canvas.create_rectangle(x1, y1, x2, y2, fill=color, outline="black")

                # Vẽ máy hút bụi (M)
                if i == self.current_x and j == self.current_y:
                    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
                    r = min(cell_w, cell_h) * 0.3
                    self.canvas.create_oval(cx-r, cy-r, cx+r, cy+r, fill="#4CAF50", outline="darkgreen", width=2)
                    self.canvas.create_text(cx, cy, text="M", fill="white", font=("Arial", int(r), "bold"))

    # ================= LOGIC THUẬT TOÁN =================
    def is_clean(self, room_state):
        for row in room_state:
            if 1 in row:
                return False
        return True

    def get_children(self, current_room, x, y):
        children = []
        def clean(nx, ny):
            copy_room = [list(row) for row in current_room]
            copy_room[nx][ny] = 0
            return tuple(tuple(row) for row in copy_room)

        if x > 0: children.append((clean(x-1,y), x - 1, y, "UP"))
        if x < self.m - 1: children.append((clean(x+1,y), x + 1, y, "DOWN"))
        if y > 0: children.append((clean(x,y-1), x, y - 1, "LEFT"))
        if y < self.n - 1: children.append((clean(x,y+1), x, y + 1, "RIGHT"))
        return children

    # --- BFS / DFS ---
    def bfs_1(self, start_room, x, y):
        start_room = tuple(tuple(row) for row in start_room)
        if self.is_clean(start_room): return [], 0
        frontier = deque([(start_room, x, y, [])])
        reached = set()

        while frontier:
            current_room, cx, cy, path = frontier.popleft()
            state_signature = (current_room, cx, cy)

            if self.is_clean(current_room): return path, len(reached)
            if state_signature in reached: continue
            reached.add(state_signature)

            for n_room, nx, ny, action in self.get_children(current_room, cx, cy):
                if (n_room, nx, ny) not in reached:
                    frontier.append((n_room, nx, ny, path + [action]))
        return None, len(reached)

    def bfs_2(self, start_room, x, y):
        start_room = tuple(tuple(row) for row in start_room)
        if self.is_clean(start_room): return [], 0
        frontier = deque([(start_room, x, y, [])])
        reached = set()

        while frontier:
            current_room, cx, cy, path = frontier.popleft()
            state_signature = (current_room, cx, cy)
            if state_signature in reached: continue
            reached.add(state_signature)

            for n_room, nx, ny, action in self.get_children(current_room, cx, cy):
                child_signature = (n_room, nx, ny)
                new_path = path + [action]
                if self.is_clean(n_room):
                    reached.add(child_signature)
                    return new_path, len(reached)
                if child_signature not in reached:
                    frontier.append((n_room, nx, ny, new_path))
        return None, len(reached)

    def dfs_1(self, start_room, x, y):
        start_room = tuple(tuple(row) for row in start_room)
        if self.is_clean(start_room): return [], 0
        frontier = deque([(start_room, x, y, [])])
        reached = set()

        while frontier:
            current_room, cx, cy, path = frontier.pop()
            state_signature = (current_room, cx, cy)

            if self.is_clean(current_room): return path, len(reached)
            if state_signature in reached: continue
            reached.add(state_signature)

            for n_room, nx, ny, action in self.get_children(current_room, cx, cy):
                if (n_room, nx, ny) not in reached:
                    frontier.append((n_room, nx, ny, path + [action]))
        return None, len(reached)

    def dfs_2(self, start_room, x, y):
        start_room = tuple(tuple(row) for row in start_room)
        if self.is_clean(start_room): return [], 0
        frontier = deque([(start_room, x, y, [])])
        reached = set()

        while frontier:
            current_room, cx, cy, path = frontier.pop()
            state_signature = (current_room, cx, cy)
            if state_signature in reached: continue
            reached.add(state_signature)

            for n_room, nx, ny, action in self.get_children(current_room, cx, cy):
                child_signature = (n_room, nx, ny)
                new_path = path + [action]
                if self.is_clean(n_room):
                    reached.add(child_signature)
                    return new_path, len(reached)
                if child_signature not in reached:
                    frontier.append((n_room, nx, ny, new_path))
        return None, len(reached)

    # --- IDS LOẠI 1 ---
    def ids_1(self, start_room, x, y):
        start_room = tuple(tuple(row) for row in start_room)
        total_pop = 0
        for depth in range(0, 100):
            result, p_count = self.dls_1(start_room, x, y, depth)
            total_pop += p_count
            if result != CUTOFF:
                if result == FAILURE: return None, total_pop
                return result, total_pop
        return None, total_pop

    def dls_1(self, start_room, x, y, limit):
        frontier = deque([(start_room, x, y, [], 0, frozenset([(start_room, x, y)]))])
        result = FAILURE
        p_count = 0

        while frontier:
            current_room, current_x, current_y, path, depth, ancestors = frontier.pop()
            p_count += 1

            if self.is_clean(current_room): return path, p_count

            if depth >= limit:
                result = CUTOFF
            else:
                for next_room, next_x, next_y, action in self.get_children(current_room, current_x, current_y):
                    child_signature = (next_room, next_x, next_y)
                    if child_signature not in ancestors:
                        new_path = path + [action]
                        new_ancestors = frozenset(ancestors | {child_signature})
                        frontier.append((next_room, next_x, next_y, new_path, depth + 1, new_ancestors))
        return result, p_count

    # --- IDS LOẠI 2 ---
    def ids_2(self, start_room, x, y):
        start_room = tuple(tuple(row) for row in start_room)
        total_pop = 0
        for depth in range(0, 100):
            result, p_count = self.dls_2(start_room, x, y, depth)
            total_pop += p_count
            if result != CUTOFF:
                if result == FAILURE: return None, total_pop
                return result, total_pop
        return None, total_pop

    def dls_2(self, start_room, x, y, limit):
        if self.is_clean(start_room): return [], 0
        frontier = deque([(start_room, x, y, [], 0, frozenset([(start_room, x, y)]))])
        result = FAILURE
        p_count = 0

        while frontier:
            current_room, current_x, current_y, path, depth, ancestors = frontier.pop()
            p_count += 1

            if depth >= limit:
                result = CUTOFF
            else:
                for next_room, next_x, next_y, action in self.get_children(current_room, current_x, current_y):
                    child_signature = (next_room, next_x, next_y)
                    if child_signature not in ancestors:
                        new_path = path + [action]
                        if self.is_clean(next_room): return new_path, p_count
                        new_ancestors = frozenset(ancestors | {child_signature})
                        frontier.append((next_room, next_x, next_y, new_path, depth + 1, new_ancestors))
        return result, p_count

    # --- UNIFORM COST SEARCH (UCS) ---
    def ucs(self, start_room, x, y):
        start_room = tuple(tuple(row) for row in start_room)
        counter = 0
        frontier = []
        heapq.heappush(frontier, (0, counter, start_room, x, y, []))
        visited = set()
        pop_count = 0

        while frontier:
            cost, _, current_room, current_x, current_y, path = heapq.heappop(frontier)
            pop_count += 1
            state_signature = (current_room, current_x, current_y)

            if self.is_clean(current_room): return path, pop_count
            if state_signature in visited: continue

            visited.add(state_signature)

            for next_room, next_x, next_y, action in self.get_children(current_room, current_x, current_y):
                child_signature = (next_room, next_x, next_y)
                if child_signature not in visited:
                    counter += 1
                    new_path = path + [action]
                    new_cost = cost + 1
                    heapq.heappush(frontier, (new_cost, counter, next_room, next_x, next_y, new_path))
        return None, pop_count

    # ================= ĐIỀU KHIỂN & MÔ PHỎNG =================
    def start_simulation(self):
        if self.is_animating: return

        # Reset phòng về trạng thái ban đầu của lần sinh gần nhất (Full rác)
        self.room = [[1 for _ in range(self.n)] for _ in range(self.m)]
        self.room[self.start_x][self.start_y] = 0
        self.current_x, self.current_y = self.start_x, self.start_y
        self.draw_grid()
        self.clear_log()

        algo_name = self.algo_var.get()
        self.log(f"--- Đang chạy: {algo_name} ---")
        self.root.update()

        # Áp dụng thuật toán được chọn
        actions, metric_count = None, 0

        if algo_name == "BFS Loại 1": actions, metric_count = self.bfs_1(self.room, self.start_x, self.start_y)
        elif algo_name == "BFS Loại 2": actions, metric_count = self.bfs_2(self.room, self.start_x, self.start_y)
        elif algo_name == "DFS Loại 1": actions, metric_count = self.dfs_1(self.room, self.start_x, self.start_y)
        elif algo_name == "DFS Loại 2": actions, metric_count = self.dfs_2(self.room, self.start_x, self.start_y)
        elif algo_name == "IDS Loại 1": actions, metric_count = self.ids_1(self.room, self.start_x, self.start_y)
        elif algo_name == "IDS Loại 2": actions, metric_count = self.ids_2(self.room, self.start_x, self.start_y)
        elif algo_name == "Uniform Cost Search (UCS)": actions, metric_count = self.ucs(self.room, self.start_x, self.start_y)

        if actions is None:
            self.lbl_result.config(text=f"Không tìm thấy đường đi! (Đã xét: {metric_count} trạng thái)", fg="red")
            self.log("Không tìm thấy đường đi!")
        else:
            path_str = " -> ".join(actions) if actions else "(Vị trí bắt đầu đã sạch)"
            result_txt = f"Tổng số bước di chuyển: {len(actions)} | Số trạng thái đã xét (Reached/Pop): {metric_count}\nCác bước: {path_str}"
            self.lbl_result.config(text=result_txt, fg="green")

            self.path = actions
            self.step_idx = 0
            self.is_animating = True
            self.log(f"Tìm thấy giải pháp ({len(actions)} bước). Bắt đầu mô phỏng...")
            self.animate_step()

    def animate_step(self):
        if not self.is_animating: return

        if self.step_idx < len(self.path):
            act = self.path[self.step_idx]
            self.step_idx += 1

            self.log(f"Bước {self.step_idx}: {act}")

            if act == "UP": self.current_x -= 1
            elif act == "DOWN": self.current_x += 1
            elif act == "LEFT": self.current_y -= 1
            elif act == "RIGHT": self.current_y += 1

            self.room[self.current_x][self.current_y] = 0 # Dọn sạch ô hiện tại
            self.draw_grid()

            # Chờ 500ms trước khi chạy bước tiếp theo
            self.animation_id = self.root.after(500, self.animate_step)
        else:
            self.is_animating = False
            self.log("--- HOÀN THÀNH MÔ PHỎNG ---")

    def stop_simulation(self):
        self.is_animating = False
        if self.animation_id:
            self.root.after_cancel(self.animation_id)
            self.animation_id = None
        self.log("Đã dừng mô phỏng.")

if __name__ == "__main__":
    root = tk.Tk()
    app = VacuumGUI(root)
    root.mainloop()